# 03 — Generator Timeseries Sintetis

**Data yang dihasilkan notebook ini SINTETIS.** Bukan data operasi
PLTU Jeranjang. Tujuannya membuat pipeline dapat dibangun dan diuji
sebelum data DCS tersedia.

Lihat `docs/LIMITATIONS.md` untuk batas pemakaiannya.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

from backend.app.core.config import get_settings
settings = get_settings()
print("Akar proyek:", settings.paths.root)
print("Mode deployment:", settings.deployment_mode)

## Membangkitkan satu bulan untuk pemeriksaan

In [ ]:
import numpy as np
from backend.app.data.event_etl import load_registry
from backend.app.data.synthetic import GenerationSpec, generate_chunk

registry = load_registry(settings)
index = pd.date_range("2024-11-01", "2024-11-30 23:59", freq="1min", name="timestamp")
spec = GenerationSpec(2024, 2024, "1min", seed=settings.synthetic_seed)
chunk = generate_chunk(index, registry, settings, spec, np.random.default_rng(spec.seed))

print(f"{len(chunk):,} baris, {chunk.shape[1]} kolom")
chunk[["timestamp", "unit_load_mw", "main_steam_flow", "main_bed_temperature",
       "bed_differential_pressure", "oxygen_o2"]].head()

## Kewajaran fisik terhadap `units.yaml`

In [ ]:
running = chunk.loc[chunk["is_running"] == 1]
rows = []
for column, bounds in settings.plausible_ranges.items():
    if column not in running.columns:
        continue
    values = running[column].dropna()
    rows.append({
        "kolom": column,
        "min": round(float(values.min()), 1),
        "median": round(float(values.median()), 1),
        "max": round(float(values.max()), 1),
        "batas": f"{bounds[0]} .. {bounds[1]}",
        "status": "OK" if values.min() >= bounds[0] and values.max() <= bounds[1] else "DI LUAR",
    })
pd.DataFrame(rows)

## Degradasi muncul sebelum event, bukan sesudahnya

In [ ]:
import matplotlib.pyplot as plt
from backend.app.reports import viz

viz.apply_theme()

frame = chunk.set_index("timestamp")
ramp = frame["ramp_furnace_agglomeration"]
window = ramp[ramp > 0]
if len(window):
    start = window.index[0] - pd.Timedelta(hours=2)
    end = window.index[-1] + pd.Timedelta(hours=2)
    view = frame.loc[start:end]

    figure, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
    axes[0].plot(view.index, view["ramp_furnace_agglomeration"], color=viz.CATEGORICAL[1])
    axes[0].set_ylabel("intensitas ramp")
    axes[1].plot(view.index, view["bed_differential_pressure"], color=viz.CATEGORICAL[0])
    axes[1].set_ylabel("bed DP, kPa")
    axes[2].plot(view.index, view["main_bed_temperature"], color=viz.CATEGORICAL[2])
    axes[2].set_ylabel("main bed, C")
    axes[0].set_title("Degradasi aglomerasi mendahului event", loc="left", fontweight="600")
    plt.tight_layout()
else:
    print("Tidak ada ramp aglomerasi pada potongan ini.")

## Kualitas data — cacat memang disengaja

In [ ]:
from backend.app.data import quality

result = quality.assess(chunk, settings)
print(result.render())

## Membangkitkan seluruh rentang

Berjalan beberapa menit dan menulis sekitar 760 MB ke
`backend/datasets/synthetic/`. Lewati bila berkasnya sudah ada.

In [ ]:
# from backend.app.data.synthetic import generate
# manifest = generate(settings, GenerationSpec(2020, 2026, "1min", settings.synthetic_seed))
# pd.DataFrame(manifest["years"])